# Module 02 — Supervised Learning
## 02-04: Gradient Boosting & AdaBoost

**Objective:** Implement AdaBoost (sequential sample re-weighting with exponential loss) and Gradient Boosting (gradient descent in function space with log-loss) from scratch using weighted decision stumps and regression trees as base learners, then compare against scikit-learn's optimized implementations.

**Prerequisites:** 2-03 (Decision Trees & Random Forests — recursive splitting, impurity measures)

## Part 0 — Setup & Prerequisites

This notebook covers **AdaBoost** and **Gradient Boosting** — two sequential ensemble methods that build models one at a time, where each new model corrects the errors of the previous ensemble. AdaBoost achieves this by *re-weighting* misclassified training samples so later stumps focus on hard examples. Gradient Boosting achieves this more generally by fitting each new tree to the *negative gradient* (pseudo-residuals) of the loss function — making it gradient descent in function space. Both use **shrinkage** (a learning rate $\nu$) to prevent any single weak learner from dominating.

We implement:
- `WeightedDecisionStump` — a depth-1 tree that respects sample weights (AdaBoost's weak learner)
- `AdaBoostClassifier` — binary classification via exponential loss minimization
- `RegressionDecisionTree` — a regression tree for fitting continuous pseudo-residuals
- `GradientBoostingClassifier` — binary classification via log-loss gradient boosting

**Prerequisites:** 2-03 (Decision Trees & Random Forests)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings("ignore")

import random
import math
import time
from typing import Generator
from dataclasses import dataclass, field
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.ensemble import (
    AdaBoostClassifier as SklearnAdaBoost,
    GradientBoostingClassifier as SklearnGBM,
    HistGradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier as SklearnDT
import sklearn

print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch:      {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Classical ML — no GPU needed
print(f"Random seed set: {SEED}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Dataset
N_SAMPLES: int = 800
N_FEATURES: int = 10
N_INFORMATIVE: int = 5
N_REDUNDANT: int = 2
TEST_SIZE: float = 0.20

# AdaBoost
N_ESTIMATORS_ADA: int = 50
LEARNING_RATE_ADA: float = 1.0

# Gradient Boosting
N_ESTIMATORS_GBM: int = 100
LEARNING_RATE_GBM: float = 0.1
GBM_MAX_DEPTH: int = 3
GBM_MIN_SAMPLES_SPLIT: int = 2

### Data Loading & EDA

Both algorithms are applied to a **binary classification** task. We use `make_classification` with 10 features (5 informative, 2 redundant, 3 noise), chosen to be realistic but not trivially separable — a single tree should underfit while boosting should improve meaningfully.

In [ ]:
# ── Generate binary classification dataset ────────────────────────────────────
X_bin, y_bin = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_classes=2,
    flip_y=0.05,        # 5% label noise — makes the task non-trivial
    random_state=SEED,
)

X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y_bin, test_size=TEST_SIZE, random_state=SEED, stratify=y_bin
)

print("Binary Classification Dataset")
print(f"  Total: {N_SAMPLES} samples | {N_FEATURES} features | 2 classes")
print(f"  Train/Test: {len(X_train)} / {len(X_test)}")
print(f"  Class dist (train): {dict(Counter(y_train))}")
print(f"  Feature range: [{X_bin.min():.2f}, {X_bin.max():.2f}]")
print(f"  Label noise: 5% (flip_y=0.05) — prevents perfect separation")

# ── EDA: feature distributions and correlation ────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
axes = axes.flatten()
colors = ["steelblue", "darkorange"]
for feat in range(N_FEATURES):
    for cls in range(2):
        axes[feat].hist(
            X_bin[y_bin == cls, feat], bins=20, alpha=0.5,
            color=colors[cls], label=f"Class {cls}", density=True,
        )
    axes[feat].set_title(f"Feature {feat}", fontsize=8)
    axes[feat].tick_params(labelsize=7)
    if feat == 0:
        axes[feat].legend(fontsize=7)
plt.suptitle("Feature Distributions by Class", fontsize=11)
plt.tight_layout()
plt.show()

# Majority-class baseline
majority_class = Counter(y_train).most_common(1)[0][0]
baseline_acc = accuracy_score(y_test, np.full(len(y_test), majority_class))
print(f"\nMajority-class baseline accuracy: {baseline_acc:.2%}")

---
## Part 1 — AdaBoost from Scratch

AdaBoost (Adaptive Boosting) builds an ensemble of **weak learners** (classifiers slightly better than random) by iteratively re-weighting the training samples. After each round, misclassified samples receive higher weights so the next weak learner focuses on the hard examples.

The mathematical foundation is **exponential loss minimization**:
$$\mathcal{L}_{\text{exp}} = \sum_{i=1}^{n} \exp\!\left(-y_i F(x_i)\right), \quad y_i \in \{-1, +1\}$$

where $F(x) = \sum_{t=1}^{T} \alpha_t h_t(x)$ is the ensemble score. At each round $t$:

1. Fit weak learner $h_t$ to minimize **weighted error**: $\epsilon_t = \sum_i w_i \cdot \mathbf{1}[h_t(x_i) \neq y_i]$
2. Compute learner weight: $\alpha_t = \ln\!\frac{1 - \epsilon_t}{\epsilon_t}$ &nbsp;&nbsp;(large when error is small)
3. Update sample weights: $w_i \leftarrow w_i \exp\!\left(-\alpha_t y_i h_t(x_i)\right)$, then normalize

Final prediction: $H(x) = \text{sign}\!\left(\sum_t \alpha_t h_t(x)\right)$

In [ ]:
# ── Visualize exponential loss vs. margin ─────────────────────────────────────
# Margin = y * F(x). Positive margin = correct. The loss penalizes negative margins.
margins = np.linspace(-3, 3, 300)
exp_loss  = np.exp(-margins)                          # AdaBoost
hinge     = np.maximum(0, 1 - margins)                # SVM (reference)
log_loss_curve  = np.log(1 + np.exp(-margins))        # Logistic regression
zero_one  = (margins < 0).astype(float)               # Ideal 0-1 loss

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(margins, exp_loss,       lw=2, label="Exponential (AdaBoost): $e^{-yF}$")
ax.plot(margins, log_loss_curve, lw=2, label="Log-loss (Gradient Boosting): $\\log(1+e^{-yF})$", linestyle="--")
ax.plot(margins, hinge,          lw=2, label="Hinge (SVM)", linestyle=":")
ax.plot(margins, zero_one,       lw=1.5, label="0-1 loss (ideal)", color="gray", linestyle="-.")
ax.axvline(0, color="black", linewidth=0.8, linestyle=":")
ax.set_xlim(-3, 3)
ax.set_ylim(-0.1, 4.5)
ax.set_xlabel("Margin $y \\cdot F(x)$")
ax.set_ylabel("Loss")
ax.set_title("Surrogate Loss Functions — How They Proxy the 0-1 Loss")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Key insight: exponential loss penalizes negative margins much more aggressively")
print("than log-loss. This makes AdaBoost very sensitive to outliers/label noise.")
print("Gradient Boosting with log-loss is more robust by penalizing gently.")

### 1.1 Weighted Decision Stump

AdaBoost's weak learner is typically a **decision stump** — a depth-1 tree that makes one threshold split. To incorporate sample weights, we replace the standard Gini impurity criterion with **weighted error**: the sum of weights of misclassified samples.

For each (feature, threshold) pair, we also test both *polarities* — predicting $+1$ to the left and $-1$ to the right, or vice versa. Because the two polarities are complementary ($\epsilon$ and $1-\epsilon$), the optimal polarity is always the one achieving $\epsilon \leq 0.5$.

In [ ]:
def find_best_weighted_split(
    X: np.ndarray,
    y_signed: np.ndarray,
    weights: np.ndarray,
) -> tuple[int, float, int, float]:
    """Search all (feature, threshold, polarity) combos for minimum weighted error.

    Polarity controls which side predicts +1 and which predicts -1:
        polarity=+1 → predict +1 when X[:, feat] <= threshold
        polarity=-1 → predict -1 when X[:, feat] <= threshold (flip)

    Both polarities are checked at each (feat, thresh) so the returned
    weighted_error is always <= 0.5 (guaranteed by choosing the better polarity).

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y_signed: Binary labels in {-1, +1}, shape (n_samples,).
        weights: Non-negative sample weights summing to 1, shape (n_samples,).

    Returns:
        Tuple (best_feat, best_thresh, best_polarity, best_weighted_error).
    """
    n_samples, n_features = X.shape
    best_error: float = float("inf")
    best_feat: int = 0
    best_thresh: float = 0.0
    best_polarity: int = 1

    for feat in range(n_features):
        column = X[:, feat]
        sorted_unique = np.sort(np.unique(column))
        if len(sorted_unique) < 2:
            continue
        thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

        for thresh in thresholds:
            left_mask = column <= thresh
            # Polarity +1: left → +1, right → -1
            pred_pos = np.where(left_mask, 1, -1)
            error_pos = float(weights[pred_pos != y_signed].sum())
            # Polarity -1: left → -1, right → +1  (error = 1 - error_pos since weights sum to 1)
            error_neg = 1.0 - error_pos

            if error_pos < best_error:
                best_error = error_pos
                best_feat = int(feat)
                best_thresh = float(thresh)
                best_polarity = 1

            if error_neg < best_error:
                best_error = error_neg
                best_feat = int(feat)
                best_thresh = float(thresh)
                best_polarity = -1

    return best_feat, best_thresh, best_polarity, best_error


class WeightedDecisionStump:
    """Binary decision stump optimized for AdaBoost sample weighting.

    Splits on one feature at one threshold, choosing the polarity that
    minimizes weighted classification error. Predicts in {-1, +1}.

    Attributes:
        feature_:       Feature index used for splitting.
        threshold_:     Split threshold value.
        polarity_:      +1 or -1 (determines which side predicts +1).
        weighted_error_: Weighted error on training data after fitting.
    """

    def __init__(self) -> None:
        """Initialize WeightedDecisionStump with unset split attributes."""
        self.feature_: int = 0
        self.threshold_: float = 0.0
        self.polarity_: int = 1
        self.weighted_error_: float = 0.5

    def fit(
        self,
        X: np.ndarray,
        y_signed: np.ndarray,
        weights: np.ndarray,
    ) -> "WeightedDecisionStump":
        """Fit the stump by minimizing weighted classification error.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y_signed: Labels in {-1, +1}, shape (n_samples,).
            weights: Sample weights summing to 1, shape (n_samples,).

        Returns:
            self — the fitted stump.
        """
        feat, thresh, polarity, error = find_best_weighted_split(X, y_signed, weights)
        self.feature_ = feat
        self.threshold_ = thresh
        self.polarity_ = polarity
        self.weighted_error_ = error
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict {-1, +1} labels for each sample.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted labels in {-1, +1}, shape (n_samples,).
        """
        left_pred = self.polarity_
        right_pred = -self.polarity_
        return np.where(X[:, self.feature_] <= self.threshold_, left_pred, right_pred)

    def __repr__(self) -> str:
        return (
            f"WeightedDecisionStump(feat={self.feature_}, "
            f"thresh={self.threshold_:.3f}, polarity={self.polarity_})"
        )


# ── Quick verification ────────────────────────────────────────────────────────
X_stump_test = np.array([[1.0], [2.0], [8.0], [9.0]])
y_stump_test = np.array([-1, -1, 1, 1])
uniform_weights = np.array([0.25, 0.25, 0.25, 0.25])

stump_test = WeightedDecisionStump()
stump_test.fit(X_stump_test, y_stump_test, uniform_weights)
print("Stump sanity check:")
print(f"  {stump_test}")
print(f"  Predictions: {stump_test.predict(X_stump_test)}   (expected [-1, -1, 1, 1])")
print(f"  Weighted error: {stump_test.weighted_error_:.4f}   (expected 0.0 — perfectly separable)")

### 1.2 AdaBoost Training Loop

Each round, we fit a weighted stump, compute its weight $\alpha_t$, update sample weights, and normalize. Misclassified samples receive weight multiplied by $e^{\alpha_t}$ (increased); correctly classified samples receive weight multiplied by $e^{-\alpha_t}$ (decreased). After normalization, the boosted samples dominate the next round's loss.

Edge cases to handle:
- $\epsilon_t = 0$: stump is perfect — $\alpha_t \to \infty$. We clip $\epsilon_t$ to a small positive value.
- $\epsilon_t > 0.5$: This can't happen if polarity is optimized correctly (we always pick the better polarity). Included as a safety check.

In [ ]:
class AdaBoostClassifier:
    """AdaBoost binary classifier using sequential exponential loss minimization.

    At each round, fits a WeightedDecisionStump to minimize weighted error,
    assigns it a weight alpha proportional to its accuracy, then re-weights
    samples so the next stump focuses on currently hard examples.

    Attributes:
        n_estimators:    Number of boosting rounds.
        learning_rate:   Shrinkage factor multiplying each alpha_t.
        stumps_:         List of fitted WeightedDecisionStump objects.
        alphas_:         List of learner weights alpha_t.
        train_errors_:   Per-round weighted training error.
        ensemble_train_accs_: Ensemble accuracy after each round.
    """

    def __init__(
        self,
        n_estimators: int = 50,
        learning_rate: float = 1.0,
    ) -> None:
        """Initialize AdaBoostClassifier with boosting hyperparameters.

        Args:
            n_estimators: Number of boosting rounds (weak learners).
            learning_rate: Shrinkage applied to each alpha_t. Values < 1
                slow learning, allowing more rounds before overfitting.
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.stumps_: list[WeightedDecisionStump] = []
        self.alphas_: list[float] = []
        self.train_errors_: list[float] = []
        self.ensemble_train_accs_: list[float] = []

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
    ) -> "AdaBoostClassifier":
        """Fit AdaBoost by sequential stump training with weight updates.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Binary labels in {0, 1}, shape (n_samples,).

        Returns:
            self — the fitted classifier.
        """
        n_samples = len(y)
        # Convert {0, 1} labels to {-1, +1} for the exponential loss formulation
        y_signed = 2 * y - 1
        # Initialize uniform sample weights
        weights = np.ones(n_samples) / n_samples

        self.stumps_ = []
        self.alphas_ = []
        self.train_errors_ = []
        self.ensemble_train_accs_ = []

        # Running ensemble score for tracking training accuracy
        F_train = np.zeros(n_samples)

        for round_idx in range(self.n_estimators):
            # ── Step 1: Fit weighted stump ─────────────────────────────────
            stump = WeightedDecisionStump()
            stump.fit(X, y_signed, weights)
            epsilon = stump.weighted_error_
            self.train_errors_.append(epsilon)

            # Safety clip: epsilon=0 gives alpha=inf; epsilon>=0.5 means stump
            # is no better than random (shouldn't happen with polarity optimization)
            epsilon = np.clip(epsilon, 1e-10, 1.0 - 1e-10)

            # ── Step 2: Compute learner weight alpha_t ─────────────────────
            alpha_t = self.learning_rate * np.log((1.0 - epsilon) / epsilon)

            # ── Step 3: Update sample weights ─────────────────────────────
            stump_preds = stump.predict(X)
            # Correct → -alpha_t (weight decreases), Wrong → +alpha_t (weight increases)
            weights = weights * np.exp(-alpha_t * y_signed * stump_preds)
            # ── Step 4: Normalize weights to sum to 1 ─────────────────────
            weights /= weights.sum()

            self.stumps_.append(stump)
            self.alphas_.append(float(alpha_t))

            # Track ensemble training accuracy after this round
            F_train += alpha_t * stump_preds
            ensemble_preds = np.where(F_train >= 0, 1, 0)
            self.ensemble_train_accs_.append(float(np.mean(ensemble_preds == y)))

        return self

    def _decision_function(self, X: np.ndarray) -> np.ndarray:
        """Compute the raw ensemble score (sum of weighted stump outputs).

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Ensemble scores, shape (n_samples,).
        """
        F = np.zeros(len(X))
        for alpha, stump in zip(self.alphas_, self.stumps_):
            F += alpha * stump.predict(X)
        return F

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict {0, 1} class labels from the ensemble score.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted labels in {0, 1}, shape (n_samples,).
        """
        return (self._decision_function(X) >= 0).astype(int)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities using a sigmoid of the ensemble score.

        Note: This is an approximation. AdaBoost scores are not calibrated
        probabilities, but the sigmoid provides a monotone confidence measure.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Probability matrix, shape (n_samples, 2).
        """
        F = self._decision_function(X)
        # Sigmoid calibration: p(y=1) = 1 / (1 + exp(-2F))
        p_pos = 1.0 / (1.0 + np.exp(-2.0 * np.clip(F, -500, 500)))
        return np.column_stack([1.0 - p_pos, p_pos])

    def staged_predict(self, X: np.ndarray) -> Generator[np.ndarray, None, None]:
        """Yield class predictions after each successive boosting round.

        Useful for computing boosting curves (accuracy vs. n_rounds)
        without re-fitting the model.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Yields:
            Predicted label array of shape (n_samples,) after each round.
        """
        F = np.zeros(len(X))
        for alpha, stump in zip(self.alphas_, self.stumps_):
            F += alpha * stump.predict(X)
            yield (F >= 0).astype(int)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels in {0, 1}, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def __repr__(self) -> str:
        return (
            f"AdaBoostClassifier("
            f"n_estimators={self.n_estimators}, "
            f"learning_rate={self.learning_rate})"
        )

In [ ]:
# ── Visualize how sample weights evolve across AdaBoost rounds ────────────────
# Train a small AdaBoost to track weight trajectories
N_TRACK_ROUNDS = 6
N_TRACK_SAMPLES = 40  # subset for visualization clarity

X_track = X_train[:N_TRACK_SAMPLES]
y_track = y_train[:N_TRACK_SAMPLES]
y_signed_track = 2 * y_track - 1
weights_history: list[np.ndarray] = []
alpha_history: list[float] = []

weights_track = np.ones(N_TRACK_SAMPLES) / N_TRACK_SAMPLES
weights_history.append(weights_track.copy())

for r in range(N_TRACK_ROUNDS):
    stump_r = WeightedDecisionStump()
    stump_r.fit(X_track, y_signed_track, weights_track)
    epsilon_r = np.clip(stump_r.weighted_error_, 1e-10, 1 - 1e-10)
    alpha_r = np.log((1.0 - epsilon_r) / epsilon_r)
    preds_r = stump_r.predict(X_track)
    weights_track = weights_track * np.exp(-alpha_r * y_signed_track * preds_r)
    weights_track /= weights_track.sum()
    weights_history.append(weights_track.copy())
    alpha_history.append(float(alpha_r))

# Plot weight distributions across rounds
fig, axes = plt.subplots(1, N_TRACK_ROUNDS + 1, figsize=(14, 3), sharey=True)
for r, (ax, w) in enumerate(zip(axes, weights_history)):
    bar_colors = ["darkorange" if y_track[i] == 1 else "steelblue" for i in range(N_TRACK_SAMPLES)]
    ax.bar(range(N_TRACK_SAMPLES), w, color=bar_colors, alpha=0.8, width=1.0)
    ax.set_title(f"Round {r}" if r == 0 else f"After round {r}", fontsize=8)
    ax.axhline(1 / N_TRACK_SAMPLES, color="gray", linestyle=":", lw=0.8)
    ax.tick_params(labelsize=7)
    if r == 0:
        ax.set_ylabel("Weight")
    if r > 0:
        ax.set_xlabel(f"$\\alpha$={alpha_history[r-1]:.2f}", fontsize=8)

axes[0].text(0.5, 0.95, "Blue=Class 0\nOrange=Class 1",
             transform=axes[0].transAxes, fontsize=7, ha="center", va="top")
plt.suptitle("AdaBoost Sample Weights — How Misclassified Samples Grow in Importance",
             fontsize=10)
plt.tight_layout()
plt.show()

print(f"Alpha values over {N_TRACK_ROUNDS} rounds: {[f'{a:.3f}' for a in alpha_history]}")
print("Note: high alpha = low error (the stump was very useful).")
print("After each round, the hardest-to-classify samples dominate the weight distribution.")

---
## Part 2 — Gradient Boosting from Scratch

Gradient Boosting frames ensemble building as **gradient descent in function space**. Instead of descending in parameter space ($\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$), we descent in the space of functions ($F \leftarrow F + \nu h$), where $h$ is the new tree fitted to the **negative gradient** of the loss at the current predictions.

For binary classification with log-loss $\mathcal{L}(y, F) = -\sum_i [y_i \log p_i + (1-y_i)\log(1-p_i)]$ where $p_i = \sigma(F_i)$:

$$\text{Pseudo-residual:} \quad r_i = -\frac{\partial \mathcal{L}}{\partial F_i} = y_i - p_i$$

At each round we fit a regression tree to the pseudo-residuals $\{(x_i, r_i)\}$, then update:
$$F_{m}(x) = F_{m-1}(x) + \nu \cdot h_m(x)$$

The pseudo-residuals are exactly the residuals of a logistic regression model — gradient boosting with log-loss is equivalent to fitting logistic regression iteratively via regression trees.

### 2.1 Regression Tree (Base Learner for GBM)

In [ ]:
@dataclass
class RegressionNode:
    """A single node in a regression decision tree.

    Internal nodes store the split condition (feature_idx, threshold).
    Leaf nodes store a predicted value (mean of training targets in that leaf).

    Attributes:
        feature_idx: Feature used for splitting (None for leaves).
        threshold:   Split value — go left if X[i, feature_idx] <= threshold.
        left:        Left child node.
        right:       Right child node.
        value:       Leaf prediction = mean of targets in this leaf.
        n_samples:   Training samples reaching this node.
    """
    feature_idx: int | None = None
    threshold: float | None = None
    left: "RegressionNode | None" = field(default=None, repr=False)
    right: "RegressionNode | None" = field(default=None, repr=False)
    value: float | None = None
    n_samples: int = 0

    @property
    def is_leaf(self) -> bool:
        """Return True when this node is a leaf."""
        return self.value is not None


def build_regression_tree(
    X: np.ndarray,
    y: np.ndarray,
    depth: int,
    max_depth: int,
    min_samples_split: int,
) -> RegressionNode:
    """Recursively build a regression tree minimizing weighted MSE at each split.

    Each leaf predicts the mean of the target values (pseudo-residuals) that
    reach it. The splitting criterion is total MSE weighted by child sizes:
        MSE_split = (n_left * Var(y_left) + n_right * Var(y_right)) / n

    Args:
        X: Feature matrix, shape (n_node_samples, n_features).
        y: Continuous target values (pseudo-residuals), shape (n_node_samples,).
        depth: Current depth (root = 0).
        max_depth: Maximum allowed depth before forcing a leaf.
        min_samples_split: Minimum samples required to attempt a split.

    Returns:
        A RegressionNode representing this subtree.
    """
    n_samples = len(y)
    node = RegressionNode(n_samples=n_samples)

    # Stopping conditions → leaf node predicts mean
    if depth >= max_depth or n_samples < min_samples_split or n_samples == 1:
        node.value = float(y.mean()) if n_samples > 0 else 0.0
        return node

    # Find the split minimizing weighted MSE
    best_score: float = float("inf")
    best_feat: int = -1
    best_thresh: float = 0.0

    for feat in range(X.shape[1]):
        column = X[:, feat]
        sorted_unique = np.sort(np.unique(column))
        if len(sorted_unique) < 2:
            continue
        thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

        for thresh in thresholds:
            left_mask = column <= thresh
            right_mask = ~left_mask
            n_left = left_mask.sum()
            n_right = right_mask.sum()
            if n_left == 0 or n_right == 0:
                continue
            # Weighted sum of variances (proportional to total MSE)
            mse_split = (
                n_left * float(np.var(y[left_mask])) +
                n_right * float(np.var(y[right_mask]))
            ) / n_samples
            if mse_split < best_score:
                best_score = mse_split
                best_feat = int(feat)
                best_thresh = float(thresh)

    # No valid split found → leaf
    if best_feat == -1:
        node.value = float(y.mean())
        return node

    node.feature_idx = best_feat
    node.threshold = best_thresh
    left_mask = X[:, best_feat] <= best_thresh
    node.left = build_regression_tree(
        X[left_mask], y[left_mask], depth + 1, max_depth, min_samples_split
    )
    node.right = build_regression_tree(
        X[~left_mask], y[~left_mask], depth + 1, max_depth, min_samples_split
    )
    return node


def predict_regression_sample(node: RegressionNode, x: np.ndarray) -> float:
    """Traverse the regression tree to predict a continuous value for one sample.

    Args:
        node: Current (or root) RegressionNode.
        x: Feature vector, shape (n_features,).

    Returns:
        Predicted continuous value (mean of training targets at the leaf).
    """
    if node.is_leaf:
        return node.value  # type: ignore[return-value]
    if x[node.feature_idx] <= node.threshold:  # type: ignore[index]
        return predict_regression_sample(node.left, x)   # type: ignore[arg-type]
    return predict_regression_sample(node.right, x)      # type: ignore[arg-type]

In [ ]:
class RegressionDecisionTree:
    """Regression tree for fitting continuous pseudo-residuals in Gradient Boosting.

    Splits are chosen to minimize weighted MSE (variance) of the target values.
    Each leaf predicts the mean of the targets (pseudo-residuals) that reach it.

    Attributes:
        max_depth:          Maximum depth of the tree.
        min_samples_split:  Minimum samples to attempt a split.
        root_:              Root RegressionNode after fitting.
        n_features_:        Number of input features.
        feature_importances_: Fraction of splits using each feature.
    """

    def __init__(
        self,
        max_depth: int = 3,
        min_samples_split: int = 2,
    ) -> None:
        """Initialize RegressionDecisionTree with splitting hyperparameters.

        Args:
            max_depth: Maximum depth before forcing leaf nodes.
            min_samples_split: Minimum samples required to split a node.
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root_: RegressionNode | None = None
        self.n_features_: int = 0
        self.feature_importances_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "RegressionDecisionTree":
        """Fit the regression tree to (X, pseudo-residuals).

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Continuous targets (pseudo-residuals), shape (n_samples,).

        Returns:
            self — the fitted tree.
        """
        self.n_features_ = X.shape[1]
        self.root_ = build_regression_tree(
            X, y, depth=0, max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
        )
        self.feature_importances_ = self._count_split_usage()
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict continuous values for each sample.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted values, shape (n_samples,).
        """
        if self.root_ is None:
            raise RuntimeError("Call fit() before predict().")
        return np.array([predict_regression_sample(self.root_, x) for x in X])

    def _count_split_usage(self) -> np.ndarray:
        """Compute fraction of splits using each feature (split-count importance).

        Returns:
            Array of shape (n_features,) summing to 1 (or all zeros if no splits).
        """
        counts = np.zeros(self.n_features_)

        def _traverse(node: RegressionNode) -> None:
            """Recursively count splits per feature."""
            if node.is_leaf or node.feature_idx is None:
                return
            counts[node.feature_idx] += 1
            if node.left is not None:
                _traverse(node.left)
            if node.right is not None:
                _traverse(node.right)

        if self.root_ is not None:
            _traverse(self.root_)
        total = counts.sum()
        return counts / total if total > 0 else counts

    def __repr__(self) -> str:
        return f"RegressionDecisionTree(max_depth={self.max_depth})"


# ── Regression tree sanity check ──────────────────────────────────────────────
X_reg_test = np.array([[0.0], [1.0], [2.0], [8.0], [9.0], [10.0]])
y_reg_test = np.array([-1.0, -0.9, -1.1, 0.9, 1.0, 1.1])  # two clusters
reg_tree_test = RegressionDecisionTree(max_depth=1)
reg_tree_test.fit(X_reg_test, y_reg_test)
preds_reg = reg_tree_test.predict(X_reg_test)
print("Regression Tree Sanity Check:")
print(f"  True values:  {y_reg_test}")
print(f"  Predictions:  {preds_reg.round(2)}")
print(f"  Left-cluster mean: ~{y_reg_test[:3].mean():.2f}, Right-cluster mean: ~{y_reg_test[3:].mean():.2f}")
print(f"  MSE: {np.mean((preds_reg - y_reg_test)**2):.4f}  (should be near 0 for depth-1 on this data)")

### 2.2 Gradient Boosting Classifier

The key components:
- **Initialization:** $F_0 = \log\!\frac{\bar{y}}{1-\bar{y}}$ (log-odds of the training class proportion)
- **Pseudo-residuals:** $r_i = y_i - \sigma(F_i)$ (negative gradient of log-loss)
- **Update:** $F_{m}(x) = F_{m-1}(x) + \nu \cdot h_m(x)$ where $h_m$ is fitted to $\{r_i\}$
- **Prediction:** $\hat{p} = \sigma(F_T(x))$, class = argmax

Leaf values use the simple mean of pseudo-residuals (the exact Newton-Raphson update $\sum r_i / \sum p_i(1-p_i)$ is mentioned in the Going Further section).

In [ ]:
def _sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid function.

    Uses separate formulas for positive and negative x to avoid overflow.

    Args:
        x: Input array of any shape.

    Returns:
        Sigmoid output in (0, 1), same shape as x.
    """
    return np.where(x >= 0,
                    1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))


def _log_loss(y: np.ndarray, p: np.ndarray) -> float:
    """Compute binary cross-entropy loss.

    Args:
        y: True binary labels in {0, 1}, shape (n_samples,).
        p: Predicted probabilities in (0, 1), shape (n_samples,).

    Returns:
        Mean log-loss (positive scalar).
    """
    p_clipped = np.clip(p, 1e-7, 1.0 - 1e-7)
    return float(-np.mean(y * np.log(p_clipped) + (1 - y) * np.log(1 - p_clipped)))


class GradientBoostingClassifier:
    """Gradient Boosting binary classifier using log-loss and regression trees.

    Each round fits a RegressionDecisionTree to the pseudo-residuals
    (y - sigmoid(F)), then updates F by shrinkage * tree predictions.
    This implements gradient descent in function space on the log-loss objective.

    Attributes:
        n_estimators:       Number of boosting rounds.
        learning_rate:      Shrinkage factor nu.
        max_depth:          Depth of each regression tree.
        min_samples_split:  Minimum samples to split a regression tree node.
        F0_:                Initial log-odds constant (scalar).
        trees_:             List of fitted RegressionDecisionTree objects.
        train_losses_:      Log-loss on training set after each round.
        n_features_:        Number of input features.
        feature_importances_: Split-count importances averaged over all trees.
    """

    def __init__(
        self,
        n_estimators: int = 100,
        learning_rate: float = 0.1,
        max_depth: int = 3,
        min_samples_split: int = 2,
    ) -> None:
        """Initialize GradientBoostingClassifier with boosting and tree hyperparameters.

        Args:
            n_estimators: Number of regression trees to build sequentially.
            learning_rate: Shrinkage nu. Small values require more estimators
                but typically yield better generalization (nu in [0.01, 0.3]).
            max_depth: Depth of each regression tree (3-5 is typical for GBM).
            min_samples_split: Minimum samples required to split a tree node.
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.F0_: float = 0.0
        self.trees_: list[RegressionDecisionTree] = []
        self.train_losses_: list[float] = []
        self.n_features_: int = 0
        self.feature_importances_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GradientBoostingClassifier":
        """Fit the gradient boosted ensemble on training data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Binary labels in {0, 1}, shape (n_samples,).

        Returns:
            self — the fitted classifier.
        """
        self.n_features_ = X.shape[1]

        # Initialize F_0 = log-odds of positive class proportion
        p0 = float(y.mean())
        p0 = np.clip(p0, 1e-7, 1.0 - 1e-7)
        self.F0_ = float(np.log(p0 / (1.0 - p0)))

        # Working predictions F(x) for all training samples
        F = np.full(len(y), self.F0_)

        self.trees_ = []
        self.train_losses_ = []

        for m in range(self.n_estimators):
            # ── Current probabilities ──────────────────────────────────────
            p = _sigmoid(F)

            # ── Pseudo-residuals = negative gradient of log-loss ───────────
            # r_i = y_i - p_i  (identical to residuals for MSE loss)
            residuals = y.astype(float) - p

            # ── Fit regression tree to pseudo-residuals ────────────────────
            tree = RegressionDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
            )
            tree.fit(X, residuals)
            self.trees_.append(tree)

            # ── Update ensemble with shrinkage ─────────────────────────────
            F = F + self.learning_rate * tree.predict(X)

            # Track training loss after this round
            train_loss = _log_loss(y, _sigmoid(F))
            self.train_losses_.append(train_loss)

        # Aggregate feature importances (mean split-count over all trees)
        fi_list = [
            t.feature_importances_
            for t in self.trees_
            if t.feature_importances_ is not None
        ]
        if fi_list:
            self.feature_importances_ = np.mean(fi_list, axis=0)

        return self

    def _predict_raw(self, X: np.ndarray) -> np.ndarray:
        """Compute raw ensemble scores F(x) for all samples.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Raw scores, shape (n_samples,).
        """
        F = np.full(len(X), self.F0_)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
        return F

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities via sigmoid of ensemble score.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Probability matrix, shape (n_samples, 2).
        """
        p_pos = _sigmoid(self._predict_raw(X))
        return np.column_stack([1.0 - p_pos, p_pos])

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels from ensemble probabilities.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted labels in {0, 1}, shape (n_samples,).
        """
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def staged_predict_proba(self, X: np.ndarray) -> Generator[np.ndarray, None, None]:
        """Yield class probabilities after each successive boosting round.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Yields:
            Probability matrix of shape (n_samples, 2) after each round.
        """
        F = np.full(len(X), self.F0_)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
            p_pos = _sigmoid(F)
            yield np.column_stack([1.0 - p_pos, p_pos])

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels in {0, 1}, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def __repr__(self) -> str:
        return (
            f"GradientBoostingClassifier("
            f"n_estimators={self.n_estimators}, "
            f"lr={self.learning_rate}, "
            f"max_depth={self.max_depth})"
        )

### Part 2 — Sanity Checks

Before training on real data, we verify correctness on simple examples: a perfectly separable dataset (both algorithms should reach 100% accuracy quickly) and a check that GBM's log-loss decreases monotonically during training.

In [ ]:
# ── Sanity check 1: AdaBoost on perfectly separable data ─────────────────────
X_easy = np.array([[1.0, 2.0], [1.5, 2.5], [8.0, 9.0], [9.0, 8.5]])
y_easy = np.array([0, 0, 1, 1])

ada_easy = AdaBoostClassifier(n_estimators=5)
ada_easy.fit(X_easy, y_easy)

print("AdaBoost — Perfectly Separable:")
print(f"  Predictions: {ada_easy.predict(X_easy)}  (expected [0, 0, 1, 1])")
print(f"  Accuracy:    {ada_easy.score(X_easy, y_easy):.0%}  (expected 100%)")
print(f"  Round errors: {[f'{e:.4f}' for e in ada_easy.train_errors_[:3]]}  (should approach 0)")

# ── Sanity check 2: GBM on perfectly separable data ──────────────────────────
gbm_easy = GradientBoostingClassifier(n_estimators=10, learning_rate=0.5, max_depth=1)
gbm_easy.fit(X_easy, y_easy)

print("\nGradient Boosting — Perfectly Separable:")
print(f"  Predictions: {gbm_easy.predict(X_easy)}  (expected [0, 0, 1, 1])")
print(f"  Accuracy:    {gbm_easy.score(X_easy, y_easy):.0%}  (expected 100%)")
first_loss = gbm_easy.train_losses_[0]
last_loss  = gbm_easy.train_losses_[-1]
print(f"  Train loss: {first_loss:.4f} → {last_loss:.4f}  (should decrease)")
assert all(
    gbm_easy.train_losses_[i] >= gbm_easy.train_losses_[i+1] - 1e-9
    for i in range(len(gbm_easy.train_losses_)-1)
), "GBM training loss should be non-increasing!"
print("  Training loss is monotonically non-increasing.")

# ── Sanity check 3: Sigmoid is numerically stable ────────────────────────────
extreme_vals = np.array([-1000.0, -100.0, 0.0, 100.0, 1000.0])
sig_vals = _sigmoid(extreme_vals)
print(f"\nSigmoid stability: {sig_vals}  (no NaN or inf, expected [~0, ~0, 0.5, ~1, ~1])")
assert not np.any(np.isnan(sig_vals)), "Sigmoid produced NaN!"
assert not np.any(np.isinf(sig_vals)), "Sigmoid produced Inf!"

---
## Part 3 — Training on Binary make_classification

We train both algorithms on the full dataset and compare against scikit-learn's implementations. The from-scratch versions should match sklearn closely — differences arise from minor implementation details (tie-breaking, leaf value computation).

In [ ]:
# ── Train AdaBoost from scratch ───────────────────────────────────────────────
start = time.time()
ada = AdaBoostClassifier(
    n_estimators=N_ESTIMATORS_ADA,
    learning_rate=LEARNING_RATE_ADA,
)
ada.fit(X_train, y_train)
ada_time = time.time() - start

ada_train_acc = ada.score(X_train, y_train)
ada_test_acc  = ada.score(X_test,  y_test)

print("From-Scratch AdaBoost")
print(f"  Train accuracy : {ada_train_acc:.2%}")
print(f"  Test  accuracy : {ada_test_acc:.2%}")
print(f"  Fit time       : {ada_time:.2f}s")
print(f"  Stumps trained : {len(ada.stumps_)}")
print(f"  Alpha range    : [{min(ada.alphas_):.3f}, {max(ada.alphas_):.3f}]")

### Library Comparison — AdaBoost

We compare against sklearn's `AdaBoostClassifier` with `algorithm="SAMME"` (the discrete version, matching our implementation) and the same `DecisionTreeClassifier(max_depth=1)` as the base learner.

In [ ]:
# ── Sklearn AdaBoost (SAMME, matching our discrete algorithm) ─────────────────
start = time.time()
sklearn_ada = SklearnAdaBoost(
    estimator=SklearnDT(max_depth=1),
    n_estimators=N_ESTIMATORS_ADA,
    learning_rate=LEARNING_RATE_ADA,
    algorithm="SAMME",
    random_state=SEED,
)
sklearn_ada.fit(X_train, y_train)
sklearn_ada_time = time.time() - start

sklearn_ada_test_acc = sklearn_ada.score(X_test, y_test)
print("Sklearn AdaBoost (SAMME)")
print(f"  Test  accuracy : {sklearn_ada_test_acc:.2%}")
print(f"  Fit time       : {sklearn_ada_time:.3f}s")
print(f"  Accuracy delta : {abs(ada_test_acc - sklearn_ada_test_acc):.4f}")

# ── Train GradientBoosting from scratch ───────────────────────────────────────
print("\nTraining Gradient Boosting from scratch (may take ~30s)...")
start = time.time()
gbm = GradientBoostingClassifier(
    n_estimators=N_ESTIMATORS_GBM,
    learning_rate=LEARNING_RATE_GBM,
    max_depth=GBM_MAX_DEPTH,
    min_samples_split=GBM_MIN_SAMPLES_SPLIT,
)
gbm.fit(X_train, y_train)
gbm_time = time.time() - start

gbm_train_acc = gbm.score(X_train, y_train)
gbm_test_acc  = gbm.score(X_test,  y_test)

print(f"From-Scratch GradientBoosting ({gbm_time:.1f}s)")
print(f"  Train accuracy : {gbm_train_acc:.2%}")
print(f"  Test  accuracy : {gbm_test_acc:.2%}")
print(f"  Final train loss: {gbm.train_losses_[-1]:.4f}")

### Library Comparison — Gradient Boosting & XGBoost Concepts

sklearn's `GradientBoostingClassifier` uses the same log-loss + regression tree strategy but with optimized C implementations and Newton-Raphson leaf values. `HistGradientBoostingClassifier` adds XGBoost-style improvements:
- **Histogram binning:** Features are discretized into bins for faster split search ($O(n_\text{bins})$ vs $O(n \log n)$)
- **L2 leaf regularization:** Shrinks leaf values to prevent overfitting
- **Sample subsampling:** Uses a fraction of data per round (reduces variance like bagging)

In [ ]:
# ── Sklearn GradientBoostingClassifier ────────────────────────────────────────
start = time.time()
sklearn_gbm = SklearnGBM(
    n_estimators=N_ESTIMATORS_GBM,
    learning_rate=LEARNING_RATE_GBM,
    max_depth=GBM_MAX_DEPTH,
    random_state=SEED,
)
sklearn_gbm.fit(X_train, y_train)
sklearn_gbm_time = time.time() - start
sklearn_gbm_test_acc = sklearn_gbm.score(X_test, y_test)

print("Sklearn GradientBoostingClassifier")
print(f"  Test  accuracy : {sklearn_gbm_test_acc:.2%}")
print(f"  Fit time       : {sklearn_gbm_time:.3f}s  (Cython optimized)")
print(f"  Accuracy delta vs scratch: {abs(gbm_test_acc - sklearn_gbm_test_acc):.4f}")

# ── HistGradientBoosting (XGBoost-style) ──────────────────────────────────────
start = time.time()
hist_gbm = HistGradientBoostingClassifier(
    max_iter=N_ESTIMATORS_GBM,
    learning_rate=LEARNING_RATE_GBM,
    max_depth=GBM_MAX_DEPTH,
    l2_regularization=0.1,   # XGBoost-style leaf weight regularization
    random_state=SEED,
)
hist_gbm.fit(X_train, y_train)
hist_gbm_time = time.time() - start
hist_gbm_test_acc = hist_gbm.score(X_test, y_test)

print("\nHistGradientBoostingClassifier (XGBoost-style)")
print(f"  Test  accuracy : {hist_gbm_test_acc:.2%}")
print(f"  Fit time       : {hist_gbm_time:.3f}s")
print("")
print("XGBoost innovations over standard GBM:")
print("  1. Histogram binning    — O(n_bins) split search instead of O(n log n)")
print("  2. L2 regularization    — shrinks leaf values: gamma = sum(r) / (sum(p*(1-p)) + lambda)")
print("  3. Column subsampling   — random feature subset per tree (like RF feature sampling)")
print("  4. Row subsampling      — stochastic GBM reduces variance, speeds training")
print("  5. Second-order updates — Newton-Raphson step uses both gradient and Hessian")

---
## Part 4 — Evaluation & Analysis

We compare all models in a summary table, plot boosting curves (test accuracy vs. number of rounds), run ablation studies on learning rate and tree depth, visualize feature importances from GBM, and examine confident misclassifications.

In [ ]:
# ── Results Summary Table ─────────────────────────────────────────────────────
baseline_preds = np.full(len(y_test), majority_class)
baseline_f1    = f1_score(y_test, baseline_preds, average="binary", zero_division=0)

single_dt = SklearnDT(max_depth=GBM_MAX_DEPTH, random_state=SEED)
single_dt.fit(X_train, y_train)
single_dt_acc = single_dt.score(X_test, y_test)
single_dt_f1  = f1_score(y_test, single_dt.predict(X_test), average="binary")

models_results = [
    ("Majority-Class Baseline",          baseline_acc,         baseline_f1),
    ("Single Decision Tree (sklearn)",    single_dt_acc,        single_dt_f1),
    ("AdaBoost         (scratch)",        ada_test_acc,         f1_score(y_test, ada.predict(X_test), average="binary")),
    ("AdaBoost SAMME   (sklearn)",        sklearn_ada_test_acc, f1_score(y_test, sklearn_ada.predict(X_test), average="binary")),
    ("GradientBoosting (scratch)",        gbm_test_acc,         f1_score(y_test, gbm.predict(X_test), average="binary")),
    ("GradientBoosting (sklearn)",        sklearn_gbm_test_acc, f1_score(y_test, sklearn_gbm.predict(X_test), average="binary")),
    ("HistGBM/XGBoost  (sklearn)",        hist_gbm_test_acc,    f1_score(y_test, hist_gbm.predict(X_test), average="binary")),
]

results_df = pd.DataFrame(
    models_results, columns=["Model", "Test Accuracy", "F1 (Binary)"]
)
results_df["Test Accuracy"] = results_df["Test Accuracy"].map("{:.2%}".format)
results_df["F1 (Binary)"]   = results_df["F1 (Binary)"].map("{:.4f}".format)
print(results_df.to_string(index=False))
print(f"\nNote: 5% label noise (flip_y=0.05) creates an accuracy ceiling near ~95%.")

### Boosting Curves — Accuracy vs. Number of Rounds

One advantage of boosted ensembles is that we can trace how performance evolves as rounds are added without re-training. The `staged_predict` / `staged_predict_proba` generators replay the training sequence on new data. This reveals:
- Whether more rounds are still helping (underfitting) or starting to hurt (overfitting)
- Where the train-test accuracy gap begins to widen

In [ ]:
# ── Boosting Curves ────────────────────────────────────────────────────────────
# AdaBoost: staged predictions
ada_test_accs_staged:  list[float] = []
ada_train_accs_staged: list[float] = ada.ensemble_train_accs_

for preds_at_round in ada.staged_predict(X_test):
    ada_test_accs_staged.append(float(np.mean(preds_at_round == y_test)))

# GBM: staged predictions
gbm_test_accs_staged:  list[float] = []
gbm_train_accs_staged: list[float] = []

for proba_at_round in gbm.staged_predict_proba(X_train):
    gbm_train_accs_staged.append(float(np.mean((proba_at_round[:, 1] >= 0.5).astype(int) == y_train)))

for proba_at_round in gbm.staged_predict_proba(X_test):
    gbm_test_accs_staged.append(float(np.mean((proba_at_round[:, 1] >= 0.5).astype(int) == y_test)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# AdaBoost curve
ada_rounds = range(1, len(ada_train_accs_staged) + 1)
axes[0].plot(ada_rounds, ada_train_accs_staged, label="Train", color="steelblue")
axes[0].plot(ada_rounds, ada_test_accs_staged,  label="Test",  color="darkorange", linestyle="--")
axes[0].axhline(baseline_acc, color="gray", linestyle=":", alpha=0.7, label=f"Baseline ({baseline_acc:.2%})")
axes[0].set_xlabel("Number of stumps")
axes[0].set_ylabel("Accuracy")
axes[0].set_title(f"AdaBoost Boosting Curve (lr={LEARNING_RATE_ADA})")
axes[0].legend()
axes[0].set_ylim(0.5, 1.01)

# GBM curve
gbm_rounds = range(1, len(gbm_train_accs_staged) + 1)
axes[1].plot(gbm_rounds, gbm_train_accs_staged, label="Train", color="steelblue")
axes[1].plot(gbm_rounds, gbm_test_accs_staged,  label="Test",  color="darkorange", linestyle="--")
axes[1].axhline(baseline_acc, color="gray", linestyle=":", alpha=0.7, label=f"Baseline ({baseline_acc:.2%})")
axes[1].set_xlabel("Number of trees")
axes[1].set_ylabel("Accuracy")
axes[1].set_title(f"GradientBoosting Boosting Curve (lr={LEARNING_RATE_GBM})")
axes[1].legend()
axes[1].set_ylim(0.5, 1.01)

plt.tight_layout()
plt.show()

best_ada_round = int(np.argmax(ada_test_accs_staged)) + 1
best_gbm_round = int(np.argmax(gbm_test_accs_staged)) + 1
print(f"Best AdaBoost test accuracy: {max(ada_test_accs_staged):.2%} at round {best_ada_round}")
print(f"Best GBM test accuracy:      {max(gbm_test_accs_staged):.2%} at round {best_gbm_round}")
print("Note: GBM with small learning rate converges slowly but more steadily than AdaBoost.")

### Ablation Study — Learning Rate & Tree Depth

**Learning rate (shrinkage $\nu$):** Small learning rate means each tree contributes less to the ensemble, requiring more rounds but achieving better generalization (the bias-variance frontier shifts). This is the classic trade-off: $\nu \downarrow$ → more estimators needed, better generalization.

**Max depth:** Deeper trees are stronger learners but more prone to overfitting. GBM with deeper trees learns faster but may overfit if not regularized. In practice, GBM uses shallow trees (depth 3–5) combined with small learning rate.

In [ ]:
# ── Learning Rate Ablation (GBM) ──────────────────────────────────────────────
learning_rates = [0.5, 0.2, 0.1, 0.05, 0.01]
lr_test_accs:  list[float] = []
lr_train_accs: list[float] = []

for lr in learning_rates:
    gbm_lr = GradientBoostingClassifier(
        n_estimators=N_ESTIMATORS_GBM,
        learning_rate=lr,
        max_depth=GBM_MAX_DEPTH,
    )
    gbm_lr.fit(X_train, y_train)
    lr_train_accs.append(gbm_lr.score(X_train, y_train))
    lr_test_accs.append(gbm_lr.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = range(len(learning_rates))
ax.plot(x_pos, lr_train_accs, "o-", label="Train", color="steelblue")
ax.plot(x_pos, lr_test_accs,  "s--", label="Test",  color="darkorange")
ax.set_xticks(x_pos)
ax.set_xticklabels([str(lr) for lr in learning_rates])
ax.set_xlabel("Learning rate $\\nu$")
ax.set_ylabel("Accuracy")
ax.set_title(f"GBM: Learning Rate vs. Accuracy ({N_ESTIMATORS_GBM} estimators)")
ax.legend()
plt.tight_layout()
plt.show()

best_lr_idx = int(np.argmax(lr_test_accs))
print(f"Best test accuracy at lr={learning_rates[best_lr_idx]}: {lr_test_accs[best_lr_idx]:.2%}")
print(f"Very small lr=0.01 gives: train={lr_train_accs[-1]:.2%}, test={lr_test_accs[-1]:.2%}")
print("Small lr with more estimators (budget permitting) is generally optimal.")

In [ ]:
# ── Max Depth Ablation (GBM) ──────────────────────────────────────────────────
depths = [1, 2, 3, 5, 7]
depth_train_accs: list[float] = []
depth_test_accs:  list[float] = []

for d in depths:
    gbm_d = GradientBoostingClassifier(
        n_estimators=N_ESTIMATORS_GBM,
        learning_rate=LEARNING_RATE_GBM,
        max_depth=d,
    )
    gbm_d.fit(X_train, y_train)
    depth_train_accs.append(gbm_d.score(X_train, y_train))
    depth_test_accs.append(gbm_d.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, depth_train_accs, "o-", label="Train", color="steelblue")
ax.plot(depths, depth_test_accs,  "s--", label="Test",  color="darkorange")
best_depth_idx = int(np.argmax(depth_test_accs))
ax.axvline(depths[best_depth_idx], color="gray", linestyle=":",
           label=f"Best depth={depths[best_depth_idx]}")
ax.set_xlabel("max_depth of each tree")
ax.set_ylabel("Accuracy")
ax.set_title(f"GBM: Tree Depth vs. Accuracy (lr={LEARNING_RATE_GBM}, n={N_ESTIMATORS_GBM})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Depth 1 (stump): train={depth_train_accs[0]:.2%}, test={depth_test_accs[0]:.2%}")
print(f"Depth {depths[best_depth_idx]} (best):  train={depth_train_accs[best_depth_idx]:.2%}, test={depth_test_accs[best_depth_idx]:.2%}")
print(f"Depth 7 (deep) : train={depth_train_accs[-1]:.2%}, test={depth_test_accs[-1]:.2%}")
print("Increasing depth speeds convergence but raises overfitting risk.")

### Feature Importances — GBM Split-Count Importances

GBM feature importances measure how often each feature is used as a split across all trees in the ensemble, normalized to sum to 1. Features used in more splits tend to be more discriminative. This is averaged across all `N_ESTIMATORS_GBM` regression trees.

In [ ]:
# ── Feature Importances (GBM vs. sklearn GBM) ─────────────────────────────────
gbm_fi      = gbm.feature_importances_
sklearn_fi  = sklearn_gbm.feature_importances_

feature_names = [f"Feature {i}" for i in range(N_FEATURES)]
sorted_idx = np.argsort(gbm_fi)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_pos = np.arange(N_FEATURES)
width = 0.35

axes[0].bar(x_pos - width/2, gbm_fi[sorted_idx],     width, label="Scratch", color="steelblue",  alpha=0.85)
axes[0].bar(x_pos + width/2, sklearn_fi[sorted_idx], width, label="Sklearn", color="darkorange", alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([feature_names[i] for i in sorted_idx], rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("Feature Importance")
axes[0].set_title("GBM Feature Importances (scratch vs sklearn)")
axes[0].legend()

# Cumulative importance plot
cum_importance = np.cumsum(np.sort(gbm_fi)[::-1])
axes[1].plot(range(1, N_FEATURES + 1), cum_importance, "o-", color="steelblue")
axes[1].axhline(0.8, color="gray", linestyle=":", label="80% threshold")
n_for_80 = int(np.searchsorted(cum_importance, 0.8)) + 1
axes[1].axvline(n_for_80, color="darkorange", linestyle=":", label=f"{n_for_80} features for 80%")
axes[1].set_xlabel("Number of top features")
axes[1].set_ylabel("Cumulative importance")
axes[1].set_title("Cumulative Feature Importance (GBM)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Top-3 features (GBM): {[feature_names[sorted_idx[i]] for i in range(3)]}")
print(f"  → importances: {gbm_fi[sorted_idx[:3]].round(3)}")
print(f"{n_for_80} features capture 80% of total importance.")
print(f"Informative features configured: {N_INFORMATIVE} / {N_FEATURES}")
print("Split-count importances roughly identify the informative features.")

### Error Analysis

We examine GBM's misclassifications on the test set. High-confidence errors — samples where the model assigns >80% probability to the wrong class — indicate either mislabeled examples (recall we added 5% label noise) or samples in genuinely ambiguous feature-space regions.

In [ ]:
# ── Error Analysis ────────────────────────────────────────────────────────────
y_pred_gbm  = gbm.predict(X_test)
y_proba_gbm = gbm.predict_proba(X_test)
misclassified = y_pred_gbm != y_test
max_confidence = y_proba_gbm.max(axis=1)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_gbm)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("GBM — Confusion Matrix")

# Confidence distribution: correct vs. incorrect
conf_correct   = max_confidence[~misclassified]
conf_incorrect = max_confidence[misclassified]
axes[1].hist(conf_correct,   bins=20, alpha=0.6, label="Correct",         color="steelblue", density=True)
axes[1].hist(conf_incorrect, bins=20, alpha=0.6, label="Misclassified",   color="darkorange", density=True)
axes[1].axvline(0.5, color="gray", linestyle=":", alpha=0.7)
axes[1].set_xlabel("Predicted Confidence (max probability)")
axes[1].set_ylabel("Density")
axes[1].set_title("Confidence Distribution: Correct vs. Misclassified")
axes[1].legend()

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_gbm, target_names=["Class 0", "Class 1"]))

# Most confident errors
error_indices = np.where(misclassified)[0]
if len(error_indices) > 0:
    sorted_by_conf = error_indices[np.argsort(max_confidence[error_indices])[::-1]]
    n_show = min(6, len(sorted_by_conf))
    print(f"Top-{n_show} Most Confident Errors (GBM):")
    print(f"  {'Idx':>6} | {'True':>5} | {'Pred':>5} | {'Confidence':>11}")
    print("  " + "-" * 36)
    for idx in sorted_by_conf[:n_show]:
        print(f"  {idx:>6} | {y_test[idx]:>5} | {y_pred_gbm[idx]:>5} | {max_confidence[idx]:>11.2%}")
    print()
    high_conf_errors = (max_confidence[misclassified] > 0.8).sum()
    print(f"High-confidence errors (>80% wrong class): {high_conf_errors} / {misclassified.sum()}")
    print("These likely include mislabeled training samples (5% flip_y) or class-boundary ambiguity.")

In [ ]:
# ── Calibration Analysis ──────────────────────────────────────────────────────
# Probability calibration: does predict_proba give reliable confidence scores?
# We use the reliability diagram (manual binning) approach.

def reliability_diagram(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute reliability diagram (calibration curve) data.

    Divides predicted probabilities into equal-width bins and computes
    the mean predicted probability and the fraction of true positives
    in each bin.

    Args:
        y_true:  True binary labels in {0, 1}, shape (n_samples,).
        y_prob:  Predicted probability for class 1, shape (n_samples,).
        n_bins:  Number of equal-width bins in [0, 1].

    Returns:
        bin_centers:  Mid-point of each bin, shape (n_filled_bins,).
        mean_pred:    Mean predicted probability per bin, shape (n_filled_bins,).
        fraction_pos: Fraction of positives per bin, shape (n_filled_bins,).
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers: list[float] = []
    mean_pred:   list[float] = []
    fraction_pos: list[float] = []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        bin_centers.append(float((lo + hi) / 2))
        mean_pred.append(float(y_prob[mask].mean()))
        fraction_pos.append(float(y_true[mask].mean()))

    return (
        np.array(bin_centers),
        np.array(mean_pred),
        np.array(fraction_pos),
    )


def compute_expected_calibration_error(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> float:
    """Compute Expected Calibration Error (ECE).

    ECE is the weighted average of the absolute difference between
    predicted confidence and empirical accuracy, weighted by bin size.

    Args:
        y_true:  True binary labels in {0, 1}, shape (n_samples,).
        y_prob:  Predicted probability for class 1, shape (n_samples,).
        n_bins:  Number of equal-width bins.

    Returns:
        ECE: Scalar in [0, 1]. Lower is better; 0 = perfect calibration.
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    n_samples = len(y_true)
    ece: float = 0.0

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        n_bin = int(mask.sum())
        if n_bin == 0:
            continue
        acc_bin = float(y_true[mask].mean())
        conf_bin = float(y_prob[mask].mean())
        ece += (n_bin / n_samples) * abs(acc_bin - conf_bin)

    return ece


# ── Compute reliability curves ────────────────────────────────────────────────
ada_proba_test = ada.predict_proba(X_test)[:, 1]
gbm_proba_test = gbm.predict_proba(X_test)[:, 1]

_, ada_mean_pred, ada_frac_pos = reliability_diagram(y_test, ada_proba_test)
_, gbm_mean_pred, gbm_frac_pos = reliability_diagram(y_test, gbm_proba_test)

ada_ece = compute_expected_calibration_error(y_test, ada_proba_test)
gbm_ece = compute_expected_calibration_error(y_test, gbm_proba_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mean_p, frac_p, ece, name, color in [
    (axes[0], ada_mean_pred, ada_frac_pos, ada_ece, "AdaBoost (scratch)", "steelblue"),
    (axes[1], gbm_mean_pred, gbm_frac_pos, gbm_ece, "GBM (scratch)", "darkorange"),
]:
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
    ax.plot(
        mean_p, frac_p, "o-", color=color, linewidth=2, markersize=7,
        label=f"{name}\nECE={ece:.4f}",
    )
    ax.fill_between(mean_p, mean_p, frac_p, alpha=0.15, color=color)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_ylabel("Fraction of Positives", fontsize=11)
    ax.set_title(f"Reliability Diagram — {name}", fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle("Probability Calibration: Reliability Diagrams", fontsize=12)
plt.tight_layout()
plt.show()

print("Expected Calibration Error (ECE) — lower is better:")
print(f"  AdaBoost (scratch): {ada_ece:.4f}")
print(f"  GBM (scratch):      {gbm_ece:.4f}")
print()
print("Note: Points above the diagonal => under-confident (predicted prob < true accuracy)")
print("Note: Points below the diagonal => over-confident (predicted prob > true accuracy)")
print("GBM scores are more calibrated because each tree is fit on log-loss pseudo-residuals.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **AdaBoost re-weights samples; Gradient Boosting fits to residuals.** Both are sequential ensemble methods, but AdaBoost uses sample weights (exponential loss), while Gradient Boosting uses pseudo-residuals (the negative gradient of any differentiable loss). GBM is strictly more general — AdaBoost is a special case of GBM with exponential loss.

2. **Shrinkage is the most important GBM hyperparameter.** Small learning rates force the ensemble to use more trees, but each tree has a smoother impact, reducing overfitting. The optimal strategy is: set $\nu$ small and increase `n_estimators` until test accuracy plateaus, using early stopping to avoid over-running.

3. **GBM pseudo-residuals are exactly logistic residuals.** For binary log-loss, the pseudo-residual at each sample is $r_i = y_i - p_i$ — the same residual you'd see in logistic regression. Gradient boosting with log-loss is thus iterative logistic regression with tree base learners.

4. **AdaBoost is more sensitive to label noise than GBM.** Exponential loss grows without bound for large negative margins, so even a few mislabeled samples can dominate the weight distribution. GBM with log-loss penalizes errors more gently, making it more robust to the 5% flip noise in our dataset.

5. **XGBoost improves GBM with regularization and second-order updates.** Newton-Raphson leaf values $\gamma_j = \frac{\sum r_i}{\sum p_i(1-p_i) + \lambda}$ are more optimal than the simple mean, and L2 regularization on leaf weights prevents individual leaves from overfitting.

### What's Next

→ **02-05 (Support Vector Machines)** introduces a fundamentally different approach: instead of building ensembles, SVMs find the single maximum-margin hyperplane that best separates classes, using the **kernel trick** to handle non-linear boundaries.

### Going Further

- **Early stopping:** Monitor test loss during training and stop when it stops improving — prevents over-boosting.
- **Newton-Raphson leaf values (XGBoost):** Replace `mean(residuals)` with `sum(r_i) / (sum(p_i*(1-p_i)) + lambda)` for better leaf optimization.
- **Friedman (2001)** — "Greedy Function Approximation: A Gradient Boosting Machine" (the canonical GBM paper)
- **Chen & Guestrin (2016)** — "XGBoost: A Scalable Tree Boosting System" (the XGBoost paper)